# Inside an encoder, block by block

A demo: how an image encoder reorganises image patches from the input of its first block to its last block,
seen through NAVE. It works with every built-in model (ViTs and ResNets).

- Every **token** of a dozen photos is one point (for ResNets, a cell of the last block's grid; earlier blocks
  are averaged onto that grid, cell by cell, following the receptive fields).
- At the **last block**, NAVE groups the tokens into concepts; each concept gets a colour, which every token
  **keeps at every earlier block**, down to the input of block 1 (`b0`).
- Each block is shown with **UMAP**, and the chosen photo's **NAVE explanation**, **uncertainty** and
  **NAVE saliency** at that block (for ResNets, saliency only exists at the last block: the classifier reads
  its channels only); for ViTs, a second row shows the same from the **attention keys**.
- Explanations are computed for **every photo** once, so changing `IMAGE` only redraws.
- A last section follows the **artifacts** of ViTs.

**Needs:** `pip install "nave-xai[animation]"`. **Downloads:** none with the default ViT-Ti/16; about 90 MB
for ViT-S/16 or ResNet50 weights, once.

In [ ]:
# %pip install "nave-xai[animation]"
import numpy as np

MODEL, TRAINING = "vitt16", "imagenet"    # any built-in model: "vits16" with "dino", "resnet50" with "imagenet" or "dino", ...
K = 8                                      # concepts at the last block
N_IMAGES = 12                              # photos used (all ship with scikit-image)
IMAGE = 2                                  # the photo shown (2: the cat); change it and re-run only the drawing cells
SECOND_ROW = "keys"                        # ViTs: the same panels from the attention keys; None for one row
UMAP_EPOCHS = 200                          # fewer = faster, rougher
SEED = 0                                   # NAVE's k-means (UMAP runs unseeded, on all cores)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def token_labels(maps, centres):
    """The concept at every patch centre of every image: one label per token."""
    return np.concatenate([m[np.ix_(centres, centres)].ravel() for m in maps])


def match_to(labels, reference, K):
    """Rename `labels` so each concept takes the id of the reference concept it overlaps most."""
    return match_colours(labels, reference, K)


def block_names(backbone):
    """The input of block 1 ("b0"), then every block."""
    return ["b0"] + list(backbone.blocks)


def pool_to(m, gh, gw):
    """Average a (C, h, w) map onto a gh x gw grid (each cell: the mean of the cells it covers)."""
    C, h, w = m.shape
    if (h, w) == (gh, gw):
        return m
    if h % gh == 0 and w % gw == 0:
        return m.reshape(C, gh, h // gh, gw, w // gw).mean((2, 4))
    from skimage.transform import resize
    return resize(m, (C, gh, gw), order=1, anti_aliasing=True, preserve_range=True)


def block_tokens(backbone, images, names):
    """For every block: the tokens of all images on the deepest block's grid, scaled to length 1.
    Returns (tokens per block, (grid height, grid width), cell size in pixels)."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    per_block = {n: [] for n in names}
    grid = None
    for img in images:
        x = ((img / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
        out = backbone(x, names)
        if grid is None:
            grid = out[names[-1]].shape[2:]
        for n in names:
            t = pool_to(out[n][0], *grid)
            per_block[n].append(t.reshape(t.shape[0], -1).T)
    tokens = [np.concatenate(per_block[n]) for n in names]
    tokens = [t / (np.linalg.norm(t, axis=1, keepdims=True) + 1e-12) for t in tokens]
    return tokens, tuple(grid), images[0].shape[0] // grid[0]


def cell_centres(size, cell):
    return np.arange(cell // 2, size, cell)


def aligned_umap(features, epochs, window=2, regularisation=0.02):
    """UMAP of a sequence of point sets (the same points), each frame kept close to its neighbours."""
    a = umap.AlignedUMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, alignment_regularisation=regularisation,
                         alignment_window_size=window)
    a.fit(features, relations=[{i: i for i in range(len(features[0]))}] * (len(features) - 1))
    return [normalise_embedding(e) for e in a.embeddings_]


def last_reference_umap(features, epochs):
    """UMAP of every frame, each starting from the last frame's layout."""
    last = umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs).fit_transform(features[-1])
    out = [umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, init=last).fit_transform(f) for f in features[:-1]]
    return [normalise_embedding(e) for e in out + [last]]


def renamed_like(labels, reference, K):
    """A renaming of concept ids so each takes the id of the reference concept it overlaps most."""
    from scipy.optimize import linear_sum_assignment
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels, reference), 1)
    r, c = linear_sum_assignment(-overlap)
    rename = np.arange(K)
    rename[r] = c
    return rename


## The photos

Twelve photos that ship with scikit-image, resized to 224 x 224 px.

In [ ]:
from skimage import data, transform

names = ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "hubble_deep_field",
         "retina", "camera", "coins", "horse", "moon", "page"][:N_IMAGES]

def load(name):
    img = getattr(data, name)()
    img = np.repeat(img[..., None], 3, axis=2) if img.ndim == 2 else img[..., :3]
    return (transform.resize(img, (224, 224), anti_aliasing=False) * 255).astype(np.uint8)

images = [load(n) for n in names]
fig, axes = plt.subplots(1, len(images), figsize=(1.3 * len(images), 1.5))
for ax, im, n in zip(np.atleast_1d(axes), images, names):
    ax.imshow(im); ax.set_title(n, fontsize=7); ax.axis("off")

### Optional: your own images

Uncomment **one** of the options below to replace the sample photos (any size; they are resized to 224 x 224 px).

In [ ]:
# from PIL import Image
# from pathlib import Path
#
# def from_file(path):
#     return np.asarray(Image.open(path).convert("RGB").resize((224, 224), Image.BICUBIC))
#
# # Option 1: every image in a folder
# paths = sorted(p for p in Path("my_images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".webp"))
# images, names = [from_file(p) for p in paths], [p.stem for p in paths]
#
# # Option 2: a single image (the concepts are then fitted on this image alone)
# images, names = [from_file("my_photo.jpg")], ["my_photo"]
#
# # Option 3: an upload button in Jupyter (ipywidgets 8); run this cell, choose files, then the next
# import ipywidgets as widgets
# upload = widgets.FileUpload(accept="image/*", multiple=True)
# display(upload)
# # ... after choosing the files, in a new cell:
# import io
# images = [np.asarray(Image.open(io.BytesIO(f["content"])).convert("RGB").resize((224, 224))) for f in upload.value]
# names = [f["name"] for f in upload.value]
#
# # Option 4: Google Colab
# from google.colab import files
# uploaded = files.upload()
# images, names = [from_file(name) for name in uploaded], list(uploaded)
#
# IMAGE = 0                                  # which of them the image panels show


## The tokens of every block

Every block's output, from `b0` (the input of block 1) to the last block, as tokens on the last block's grid,
scaled to length 1 as NAVE does; for ViTs, also the attention keys of every block.

In [ ]:
from nave_xai import NAVEImage

explainer = NAVEImage(MODEL, TRAINING, k=K, seed=SEED)
backbone = explainer.backbone
names_b = block_names(backbone)                            # "b0" (the input of block 1), then every block
is_vit = backbone.name.startswith("vit")
explainer.layers = [names_b[-1]]                           # concepts of the last block
explainer._backbone = None                                 # re-resolve with the new layers
backbone = explainer.backbone
tokens, grid, cell = block_tokens(backbone, images, names_b)
second = SECOND_ROW if (SECOND_ROW and is_vit) else None
names_2 = ["b0"] + [f"k{i}" for i in range(1, len(names_b))] if second == "keys" else None
tokens_2 = block_tokens(backbone, images, names_2)[0] if second else None
G = grid[0] * grid[1]                                      # tokens per photo
print(len(names_b), "frames:", names_b[0], "...", names_b[-1], "|", grid, "grid,", cell, "px per token")

## Colours from the last block

NAVE fits K concepts on the last block. A token's colour is the concept at its cell's centre in the
explanation map, kept at every block.

In [ ]:
explainer.fit(images)
centres = cell_centres(224, cell)
token_concept = token_labels([e.explanation for e in explainer.predict(images)], centres)
palette = explainer.palette / 255
colours = palette[token_concept]
print(np.bincount(token_concept, minlength=K), "tokens per concept")

## UMAP of every block

`AlignedUMAP` fits all blocks together, each kept close to its neighbours, so the frames are comparable and
the animation follows the gradual changes. UMAP runs unseeded here, so it can use all cores: two runs differ
slightly.

In [ ]:
import warnings
import umap

warnings.filterwarnings("ignore", module="umap")               # UMAP's notices about threads and seeds
warnings.filterwarnings("ignore", category=FutureWarning)      # scikit-learn's, inside UMAP

emb_previous = aligned_umap(tokens, UMAP_EPOCHS)          # each block kept close to its neighbours
emb_2 = aligned_umap(tokens_2, UMAP_EPOCHS) if second else None
print("embeddings ready")

## Explanations at every block, for every photo

In [ ]:
def per_block_views(layer_names, with_saliency):
    """For every block: NAVE fitted on that block alone (all photos), its concepts renamed after the last
    block's they overlap most (stable colours). Stored for every photo: explanation, uncertainty and,
    optionally, NAVE saliency, so the drawings can switch photo without recomputing."""
    views, targets = [], None
    for name in layer_names:
        nave = NAVEImage(backbone, layers=[name], k=K, seed=SEED).fit(images)
        exps = nave.predict(images, uncertainty=True)
        lab = token_labels([e.explanation for e in exps], centres)
        rename = renamed_like(lab, token_concept, K)
        v = dict(token_labels=rename[lab], token_uncertainty=token_labels([e.uncertainty for e in exps], centres),
                 explanation=[(palette[rename[e.explanation]] * 255).astype(np.uint8) for e in exps],
                 uncertainty=[e.uncertainty.astype(np.float16) for e in exps])
        if with_saliency:
            if targets is None:
                targets = [nave._predict_class(*nave._prepare(im)) for im in images]
            try:
                v["saliency"] = [nave.saliency(im, target=t).astype(np.float16) for im, t in zip(images, targets)]
            except ValueError:                 # ResNets: the classifier only reads the last block's channels
                v["saliency"] = [np.zeros(im.shape[:2], np.float16) for im in images]
        views.append(v)
    return views

per_block = per_block_views(names_b, True)
per_block_2 = per_block_views(names_2, False) if second else None
print("per-block views ready")

## A number next to the picture

How well a token's 5 nearest neighbours **at each block** predict its **final** concept.

In [ ]:
agreement = [knn_agreement(t, token_concept, seed=SEED) for t in tokens]
agreement_2 = [knn_agreement(t, token_concept, seed=SEED) for t in tokens_2] if second else None
plt.figure(figsize=(6, 2.2))
plt.plot(range(len(names_b)), agreement, marker="o", label="block outputs")
if second:
    plt.plot(range(len(names_b)), agreement_2, marker="o", label=second)
plt.xticks(range(len(names_b)), names_b, fontsize=7)
plt.xlabel("block (b0: the input of block 1)"); plt.ylabel("agreement with final concepts"); plt.ylim(0, 1)
plt.legend(fontsize=8);

## The animation

The input photo on the left; the shown photo's tokens are outlined in the UMAP.

In [ ]:
TITLES = {"umap": "UMAP of all tokens", "explanation": "NAVE explanation", "uncertainty": "uncertainty",
          "saliency": "NAVE saliency"}
PANELS = tuple(TITLES)
mine = np.arange(IMAGE * G, (IMAGE + 1) * G)               # the shown photo's tokens


def draw_panel(ax, p, emb, views, b):
    """One panel at frame b (for the slider and the first frame of the animation)."""
    if p == "umap":
        ax.scatter(*emb[b].T, c=colours, s=4, linewidths=0)
        return ax.scatter(*emb[b][mine].T, c=colours[mine], s=18, edgecolors="black", linewidths=0.4)
    if p == "explanation":
        return ax.imshow(views[b]["explanation"][IMAGE])
    if p == "uncertainty":
        return ax.imshow(views[b]["uncertainty"][IMAGE].astype(float), cmap="inferno_r", vmin=0, vmax=1)
    return ax.imshow(views[b]["saliency"][IMAGE].astype(float), cmap="seismic", vmin=-1, vmax=1)


def animate(embeddings, panels=PANELS, with_second=True, steps=6, title=""):
    global mine
    mine = np.arange(IMAGE * G, (IMAGE + 1) * G)
    panels = [p for p in panels if p in TITLES]
    rows = [("block outputs", embeddings, per_block, panels)]
    if with_second and second:
        rows.append((second, emb_2, per_block_2, [p for p in panels if p != "saliency"]))
    n_cols, n = 1 + len(panels), len(names_b)
    plt.rcParams["animation.embed_limit"] = 100
    fig = plt.figure(figsize=(2.9 * n_cols, 3.0 * len(rows) + 1.3), dpi=60)
    grid_ = fig.add_gridspec(len(rows) + 1, n_cols, height_ratios=[3] * len(rows) + [0.9], hspace=0.35)
    ax_in = fig.add_subplot(grid_[:len(rows), 0])
    ax_in.imshow(images[IMAGE]); ax_in.set_title(f"input: {names[IMAGE]}", fontsize=9); ax_in.axis("off")
    curve = fig.add_subplot(grid_[len(rows), :])
    curve.plot(range(n), agreement, color="0.4", marker="o", ms=3, label="block outputs")
    if len(rows) > 1:
        curve.plot(range(n), agreement_2, color="0.7", marker="o", ms=3, label=second)
    marker = curve.axvline(0, color="black", lw=1)
    curve.set_xticks(range(n), names_b, fontsize=6); curve.set_ylim(0, 1); curve.set_ylabel("agreement", fontsize=8)
    curve.legend(fontsize=7, loc="lower right")
    arts = []
    for r, (label, emb, views, row_panels) in enumerate(rows):
        for p in row_panels:
            ax = fig.add_subplot(grid_[r, 1 + panels.index(p)]); ax.set_xticks([]); ax.set_yticks([])
            if p == "umap":
                ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
                ax.scatter([], [])
                every = ax.scatter(*emb[0].T, c=colours, s=4, linewidths=0)
                sel = ax.scatter(*emb[0][mine].T, c=colours[mine], s=18, edgecolors="black", linewidths=0.4)
                art = (every, sel)
            else:
                art = draw_panel(ax, p, emb, views, 0)
            ax.set_title(f"{TITLES[p]} ({label})", fontsize=8)
            arts.append((p, art, emb, views))
    head = fig.suptitle("", fontsize=11)

    def frame(f):
        t = f / steps; i0, w = int(t), t - int(t); i1 = min(i0 + 1, n - 1); b = int(round(t))
        for p, art, emb, views in arts:
            if p == "umap":
                pos = (1 - w) * emb[i0] + w * emb[i1]
                art[0].set_offsets(pos); art[1].set_offsets(pos[mine])
            else:
                art.set_data(views[b][p][IMAGE].astype(float) if p != "explanation" else views[b][p][IMAGE])
        marker.set_xdata([t, t]); head.set_text(f"{title}{names_b[b]}")
        return [head, marker]

    return animation.FuncAnimation(fig, frame, frames=(n - 1) * steps + 1, interval=120, blit=False)


anim = animate(emb_previous)
display(HTML(anim.to_jshtml()))
anim.save("blocks_previous_reference.gif", writer=animation.PillowWriter(fps=8), dpi=80)
plt.close()

## With a slider

With `ipywidgets`, pick the block and the photo. Without it, the cell draws the last block.

In [ ]:
def show(block=len(names_b) - 1, image=IMAGE):
    global IMAGE, mine
    IMAGE, mine = image, np.arange(image * G, (image + 1) * G)
    emb = emb_previous
    panels = [p for p in PANELS if p in TITLES]
    rows = [("block outputs", emb, per_block, panels)] + ([(second, emb_2, per_block_2, [p for p in panels if p != "saliency"])] if second else [])
    fig, axes = plt.subplots(len(rows), len(panels) + 1, figsize=(2.9 * (len(panels) + 1), 3 * len(rows)), squeeze=False)
    for r, (label, e, views, row_panels) in enumerate(rows):
        axes[r, 0].axis("off")
        if r == 0:
            axes[r, 0].imshow(images[image]); axes[r, 0].set_title(f"input: {names[image]}", fontsize=9)
        for j, p in enumerate(panels):
            ax = axes[r, 1 + j]; ax.set_xticks([]); ax.set_yticks([])
            if p not in row_panels:
                ax.axis("off"); continue
            if p == "umap":
                ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
            draw_panel(ax, p, e, views, block)
            ax.set_title(f"{TITLES[p]} ({label})", fontsize=8)
    fig.suptitle(names_b[block], fontsize=11)
    plt.show()

try:
    import ipywidgets as widgets
    widgets.interact(show, block=widgets.IntSlider(len(names_b) - 1, 0, len(names_b) - 1, 1),
                     image=widgets.Dropdown(options=[(n, i) for i, n in enumerate(names)], value=IMAGE));
except ImportError:
    show()

## Following the artifacts (ViTs)

Large ViTs recycle some uninformative patches as storage for global information (Darcet et al., *Vision
Transformers Need Registers*). In NAVE they tend to form a concept of their own, made of the most uncertain
tokens. Three views: when each artifact token joins it for good, their uncertainty before, and the concepts
they come from.

In [ ]:
if not is_vit:
    print("Artifact tokens are a ViT phenomenon: this section is for ViT models.")
else:
    last_uncertainty = per_block[-1]["token_uncertainty"]
    mean_uncertainty = np.array([last_uncertainty[token_concept == c].mean() if np.any(token_concept == c) else 0
                                 for c in range(K)])
    ARTIFACT = int(np.argmax(mean_uncertainty))            # the most uncertain concept; set it yourself if needed
    print("artifact concept (guess):", ARTIFACT, "| mean uncertainty per concept:", np.round(mean_uncertainty, 2))
    labels = np.stack([v["token_labels"] for v in per_block])
    uncert = np.stack([v["token_uncertainty"] for v in per_block])
    art = token_concept == ARTIFACT
    stays = np.flip(np.cumprod(np.flip(labels == ARTIFACT, 0), 0), 0).astype(bool)
    flip = np.where(stays.any(0), stays.argmax(0), len(names_b))
    nb = len(names_b)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    axes[0].hist(flip[art], bins=np.arange(-0.5, nb + 0.5), color=palette[ARTIFACT], edgecolor="white")
    axes[0].set_xticks(range(nb), names_b, fontsize=6); axes[0].set_ylabel("tokens")
    axes[0].set_title("block where a token joins the artifact concept for good", fontsize=9)
    before = [uncert[:f, i] for i, f in zip(np.nonzero(art)[0], flip[art]) if f > 0]
    axes[1].plot(range(nb), uncert[:, ~art].mean(1), color="0.5", label="other tokens")
    axes[1].plot(range(nb), [np.mean([b_[k] for b_ in before if len(b_) > k]) if any(len(b_) > k for b_ in before)
                             else np.nan for k in range(nb)], color=palette[ARTIFACT], label="artifact tokens, before")
    axes[1].set_xticks(range(nb), names_b, fontsize=6); axes[1].legend(fontsize=7)
    axes[1].set_title("uncertainty before the flip", fontsize=9)
    origin = np.stack([np.bincount(labels[b][art], minlength=K) for b in range(nb)]).T
    axes[2].stackplot(range(nb), origin, colors=palette, alpha=0.9)
    axes[2].set_xticks(range(nb), names_b, fontsize=6); axes[2].set_title("their concept at each block", fontsize=9)
    fig.tight_layout()

## Things to look at

A few directions, if you want them:
- when the final concepts separate, and whether the agreement curve says the same;
- what `b0` already groups: colour, texture, position?
- block outputs against keys (ViTs);
- a ViT against a ResNet, supervised against self-supervised (`TRAINING = "dino"`);
- which concepts lose tokens to the artifacts.

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```

We thank Ulf Brefeld (Leuphana University Lüneburg) for lending us the computing resources without which this work would not have been possible.